In [1]:
!git clone https://github.com/Abhisekguha/improved-octo-giggle_VLM.git

Cloning into 'improved-octo-giggle_VLM'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 60 (delta 5), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 240.18 KiB | 7.28 MiB/s, done.
Resolving deltas: 100% (5/5), done.


In [11]:
cd /kaggle/working/improved-octo-giggle_VLM

/kaggle/working/improved-octo-giggle_VLM


In [12]:
!ls

benchmark_vlm.py       eval_run_baseModels.ipynb	   run_all.py
compare_models.py      eval_run_finetunedModels.ipynb	   run_model.py
config_finetuned.yaml  KD_pipeline.ipynb		   test_script
config.yaml	       knowledge_distillation		   vlm_bench
DOCUMENTATION.md       results_before_FT_n_KD_1438samples


In [6]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.1
!pip install --no-deps trl==0.22.2

In [13]:
!python -m knowledge_distillation.generate_teacher_labels \
    --model_path "abhi26/subCV_qwen3-8B_lora" \
    --dataset "nyu-visionx/CV-Bench" \
    --split test \
    --output knowledge_distillation/teacher_outputs/teacher_labels.jsonl \
    --features_dir knowledge_distillation/teacher_outputs/visual_features \
    --no_rationale \
    --max_samples 100

  STEP 1: Generate Teacher Labels
  Model    : abhi26/subCV_qwen3-8B_lora
  Dataset  : nyu-visionx/CV-Bench [test] → 2D only
  Output   : knowledge_distillation/teacher_outputs/teacher_labels.jsonl
  Rationale: False
  Logits   : True
README.md: 6.14kB [00:00, 2.79MB/s]
test_2d.parquet: 100%|███████████████████████| 185M/185M [00:07<00:00, 23.6MB/s]
test_3d.parquet: 100%|███████████████████████| 220M/220M [00:08<00:00, 25.5MB/s]
Filter: 100%|███████████████████████| 2638/2638 [00:11<00:00, 227.07 examples/s]
Samples: 100
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-05-29 08:25:04.837479: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780043105.065449     242 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:178

In [37]:
!cd /kaggle/working/improved-octo-giggle_VLM && git pull origin main

remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 8 (delta 6), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 2.44 KiB | 831.00 KiB/s, done.
From https://github.com/Abhisekguha/improved-octo-giggle_VLM
 * branch            main       -> FETCH_HEAD
   a964fc4..6585a8f  main       -> origin/main
Updating a964fc4..6585a8f
Fast-forward
 knowledge_distillation/config.py                 |  6 +++---
 knowledge_distillation/train_student_internvl.py | 23 +++++++++++++++++++----
 2 files changed, 22 insertions(+), 7 deletions(-)


In [38]:
!python knowledge_distillation/train_student_internvl.py \
    --student_model "OpenGVLab/InternVL3_5-1B-Instruct" \
    --teacher_labels knowledge_distillation/teacher_outputs/teacher_labels.jsonl \
    --features_dir knowledge_distillation/teacher_outputs/visual_features \
    --dataset "nyu-visionx/CV-Bench" \
    --split test \
    --kd_mode response \
    --no_rationale \
    --epochs 1 \
    --batch_size 2 \
    --lr 2e-4

2026-05-29 10:07:13.761221: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780049233.785683    1376 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780049233.793446    1376 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780049233.813575    1376 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780049233.813632    1376 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780049233.813640    1376 computation_placer.cc:177] computation placer alr

In [45]:
!cd /kaggle/working/improved-octo-giggle_VLM && git pull origin main

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 2.25 KiB | 2.25 MiB/s, done.
From https://github.com/Abhisekguha/improved-octo-giggle_VLM
 * branch            main       -> FETCH_HEAD
   3960256..b79b9fb  main       -> origin/main
Updating 3960256..b79b9fb
Fast-forward
 knowledge_distillation/evaluate_student.py | 99 +++++++++++++++++++++++++++---
 1 file changed, 92 insertions(+), 7 deletions(-)


In [ ]:
!python knowledge_distillation/evaluate_student.py \
  --student_path "OpenGVLab/InternVL3_5-1B-Instruct" \
  --adapter_path "knowledge_distillation/student_checkpoints/final" \
  --dataset "nyu-visionx/CV-Bench" \
  --split "test" \
  --max_samples 1438 \
  --output "knowledge_distillation/eval_results/student_eval_50percent.json"

  STEP 3: Evaluate Distilled Student
  Model   : OpenGVLab/InternVL3_5-1B-Instruct
  Adapter : knowledge_distillation/student_checkpoints/final
  Dataset : nyu-visionx/CV-Bench [test] → 2D only
Evaluation samples: 1438
`torch_dtype` is deprecated! Use `dtype` instead!
2026-05-29 10:20:06.721291: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780050006.743951    1573 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780050006.751441    1573 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780050006.771735    1573 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than